# International query weighting diagnostic

This notebook tests whether giving international searches more weight during training narrows the gap between domestic and international NDCG@5. It rebuilds the Supernova validation feature frames, trains one weighted RankXENDCG model, and compares it with the saved unweighted seed-42 RankXENDCG predictions from the Supernova validation run.

It skips the full ensemble, CatBoost, the meta stacker, the multi-split audit, the reranker and the final submission.

## Reuse Supernova definitions

The feature definitions live in `notebooks/supernova.ipynb`. The next cell executes four of its code cells, selected by cell id. They are the setup, the configuration, the LightGBM parameter helpers and the Supernova feature definitions. None of them train the submitted ensemble.

In [1]:
from pathlib import Path
import json

BOOT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE_NOTEBOOK = BOOT_ROOT / "notebooks" / "supernova.ipynb"
SOURCE_CELL_IDS = [
    "e93cbe69",  # Imports and repository paths
    "c0173363",  # Supernova configuration flags
    "0e61991c",  # LightGBM parameter helpers
    "dcfa3dd5",  # Supernova feature definitions
]

source_nb = json.loads(SOURCE_NOTEBOOK.read_text())
source_cells = {cell.get("id"): cell for cell in source_nb["cells"]}
missing_source_cells = [cell_id for cell_id in SOURCE_CELL_IDS if cell_id not in source_cells]
assert not missing_source_cells, f"Source notebook cells changed or missing: {missing_source_cells}"

for cell_id in SOURCE_CELL_IDS:
    source = "".join(source_cells[cell_id]["source"])
    exec(compile(source, f"{SOURCE_NOTEBOOK.name}:{cell_id}", "exec"), globals())

print(f"Loaded Supernova definitions from {SOURCE_NOTEBOOK}")

LightGBM device params: {'device': 'gpu', 'gpu_use_dp': False}
LightGBM rank gain policy: linear
Pointwise params: {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 511, 'learning_rate': 0.01, 'min_child_samples': 200, 'lambda_l1': 0.1, 'lambda_l2': 1.0, 'min_gain_to_split': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'feature_pre_filter': False, 'device': 'gpu', 'gpu_use_dp': False, 'verbose': -1, 'seed': 42, 'num_threads': -1}
Click params: {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 255, 'learning_rate': 0.025, 'min_child_samples': 150, 'lambda_l1': 0.05, 'lambda_l2': 1.0, 'feature_fraction': 0.75, 'bagging_fraction': 0.85, 'bagging_freq': 5, 'feature_pre_filter': False, 'verbose': -1, 'seed': 42, 'num_threads': -1, 'device': 'gpu', 'gpu_use_dp': False}
LambdaRank params: {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5], 'num_leaves': 511, 'learning_rate': 0.01, 'min_child_samples': 200, 'lambda_l1

## Diagnostic configuration

A query weight of `1.7` roughly equalizes domestic and international search emphasis on the validation split. The list holds that single value. Add `1.5` to it to compare a second weight.

In [2]:
DIAGNOSTIC_TAG = "supernova_bias_weighted_international"
INTERNATIONAL_QUERY_WEIGHTS = [1.7]
BASELINE_SCORE_COL = "xendcg_s42_score"
BASELINE_PREDICTIONS_PATH = OUTPUT_DIR / VALIDATION_PREDICTIONS_NAME
TRAIN_BASELINE_IF_SAVED_PREDICTIONS_MISSING = True

# Match the saved seed-42 Supernova RankXENDCG path.
DIAGNOSTIC_SEED = RANDOM_STATE
DIAGNOSTIC_NEGATIVE_DROPOUT_FRAC = XENDCG_NEGATIVE_DROPOUT_FRAC
DIAGNOSTIC_NUM_BOOST_ROUND = XENDCG_NUM_BOOST_ROUND
DIAGNOSTIC_EARLY_STOPPING = XENDCG_EARLY_STOPPING

print(f"Baseline parquet: {BASELINE_PREDICTIONS_PATH}")
print(f"International query weights: {INTERNATIONAL_QUERY_WEIGHTS}")

Baseline parquet: /home/akshitanchan/Projects/hotel-trivago-but-its-expedia/outputs/supernova_validation_predictions.parquet
International query weights: [1.7]


## Build fixed validation features

This is the slow step on a cold kernel. It reuses the original Supernova split and feature set, so the weighted model changes only the training emphasis and not the features.

In [3]:
raw_df = load_train(sample_frac=SAMPLE_FRAC if USE_SAMPLE else None, random_state=RANDOM_STATE)
print(f"Raw rows: {len(raw_df):,}  searches: {raw_df['srch_id'].nunique():,}")

train_raw, val_raw = train_val_split(raw_df, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train rows: {len(train_raw):,}  Val rows: {len(val_raw):,}")
del raw_df
gc.collect()

train_df, val_df = build_validation_feature_frames(
    train_raw,
    val_raw,
    count_reference_frame=train_raw,
    svd_components=SVD_COMPONENTS,
)
train_df, val_df = add_supernova_features(train_df, val_df)
del train_raw, val_raw
gc.collect()

assert train_df.duplicated(["srch_id", "prop_id"]).sum() == 0
assert val_df.duplicated(["srch_id", "prop_id"]).sum() == 0
feature_cols = get_feature_cols(train_df)
print(f"Features: {len(feature_cols)}")
print(f"Validation search count: {val_df['srch_id'].nunique():,}")

Raw rows: 4,958,347  searches: 199,795
Train rows: 3,966,682  Val rows: 991,665
prop_id prior unseen fill rate: 0.3513%
Expedia emulator features: kept 155 target-free columns; excluded 114 target-derived columns
First excluded target-derived emulator columns: ['click_to_book_rate_oof', 'country_pair_affinity', 'country_pair_click_rate', 'country_pair_star_book_prior', 'country_pair_star_count', 'country_pair_star_ctr_prior', 'dest_book_rate', 'dest_hotel_affinity', 'dest_hotel_affinity_diff_mean', 'dest_hotel_affinity_rank', 'dest_star_book_rate_unbiased', 'dest_star_book_rate_unbiased_diff_mean', 'dest_star_book_rate_unbiased_rank', 'dest_star_click_only_rate_unbiased', 'dest_star_click_only_rate_unbiased_diff_mean', 'dest_star_click_only_rate_unbiased_rank', 'dest_star_count_unbiased', 'dest_star_ctr_unbiased', 'dest_star_ctr_unbiased_diff_mean', 'dest_star_ctr_unbiased_rank']
Features: 273
Validation search count: 39,959


## Weighted RankXENDCG helpers

These helpers train RankXENDCG with a per-query weight for international searches and compute domestic and international metrics.

In [4]:
def query_local_negative_dropout(frame, drop_frac, seed):
    if drop_frac <= 0:
        return frame
    rng = np.random.default_rng(seed)
    relevance = make_relevance(frame)
    keep = relevance > 0
    zero_idx = np.where(~keep)[0]
    keep[zero_idx] = rng.random(len(zero_idx)) >= drop_frac
    keep_series = pd.Series(keep, index=frame.index)
    per_query_kept = keep_series.groupby(frame["srch_id"]).sum()
    empty_queries = set(per_query_kept.index[per_query_kept == 0])
    if empty_queries:
        first_rows = (
            frame[frame["srch_id"].isin(empty_queries)]
            .groupby("srch_id", sort=False)
            .head(1)
            .index
        )
        keep_series.loc[first_rows] = True
    return frame.loc[keep_series.to_numpy()].copy()


def search_segment(frame, segment_col):
    return frame.groupby("srch_id", sort=False)[segment_col].transform("first")


def rank_recall_at_5(frame, score_col):
    relevance = make_relevance(frame) > 0
    if not relevance.any():
        return np.nan
    rank = frame.groupby("srch_id")[score_col].rank(ascending=False, method="first")
    return float((rank[relevance] <= 5).mean())


def domestic_fairness_metrics(frame, score_col):
    score_frame = frame[["srch_id", "prop_id", "booking_bool", "click_bool", "is_domestic", score_col]].copy()
    search_domestic = score_frame.groupby("srch_id")["is_domestic"].first()
    result = {
        "score_col": score_col,
        "overall": {
            "ndcg5": float(compute_ndcg_fast(score_frame, score_col)),
            "recall5": rank_recall_at_5(score_frame, score_col),
            "n_searches": int(score_frame["srch_id"].nunique()),
            "n_rows": int(len(score_frame)),
        },
        "is_domestic": {},
    }
    for value in [0, 1]:
        search_ids = search_domestic.index[search_domestic == value]
        segment = score_frame[score_frame["srch_id"].isin(search_ids)]
        result["is_domestic"][f"is_domestic={value}"] = {
            "ndcg5": float(compute_ndcg_fast(segment, score_col)),
            "recall5": rank_recall_at_5(segment, score_col),
            "n_searches": int(segment["srch_id"].nunique()),
            "n_rows": int(len(segment)),
        }
    segment_ndcg = [entry["ndcg5"] for entry in result["is_domestic"].values()]
    result["is_domestic_epr"] = float(min(segment_ndcg) / max(segment_ndcg))
    return result


def train_rankxendcg_with_international_query_weight(query_weight, score_col):
    fit_df = query_local_negative_dropout(
        train_df,
        DIAGNOSTIC_NEGATIVE_DROPOUT_FRAC,
        DIAGNOSTIC_SEED,
    )
    print(f"{score_col}: dropout kept {len(fit_df):,}/{len(train_df):,} training rows")
    tr_sorted, X_tr, y_tr, group_tr = sort_for_group_model(
        fit_df,
        feature_cols,
        make_relevance(fit_df),
    )
    va_sorted, X_va, y_va, group_va = sort_for_group_model(
        val_df,
        feature_cols,
        make_relevance(val_df),
    )

    train_search_domestic = search_segment(tr_sorted, "is_domestic")
    row_weight = np.where(train_search_domestic.eq(0), query_weight, 1.0).astype("float32")
    print(
        f"{score_col}: international query weight={query_weight:.3f}; "
        f"weighted training-row mean={row_weight.mean():.4f}"
    )

    train_set = lgb.Dataset(X_tr, label=y_tr, group=group_tr, weight=row_weight)
    val_set = lgb.Dataset(X_va, label=y_va, group=group_va, reference=train_set)
    model = train_lgb(
        with_lgb_seed(XENDCG_PARAMS, DIAGNOSTIC_SEED),
        train_set,
        num_boost_round=DIAGNOSTIC_NUM_BOOST_ROUND,
        valid_sets=[val_set],
        callbacks=[
            lgb.early_stopping(DIAGNOSTIC_EARLY_STOPPING),
            lgb.log_evaluation(200),
        ],
    )
    pred = va_sorted[["srch_id", "prop_id"]].copy()
    pred[score_col] = predict_lgb_snapshot(model, X_va, DIAGNOSTIC_NUM_BOOST_ROUND).astype("float32")
    print(f"{score_col}: trees={lgb_iteration_count(model, DIAGNOSTIC_NUM_BOOST_ROUND)}")
    del fit_df, tr_sorted, va_sorted, X_tr, X_va, y_tr, y_va, group_tr, group_va, train_set, val_set
    gc.collect()
    return model, pred


def metrics_row(name, query_weight, metrics):
    international = metrics["is_domestic"]["is_domestic=0"]
    domestic = metrics["is_domestic"]["is_domestic=1"]
    return {
        "name": name,
        "international_query_weight": float(query_weight),
        "overall_ndcg5": metrics["overall"]["ndcg5"],
        "overall_recall5": metrics["overall"]["recall5"],
        "international_ndcg5": international["ndcg5"],
        "domestic_ndcg5": domestic["ndcg5"],
        "international_recall5": international["recall5"],
        "domestic_recall5": domestic["recall5"],
        "domestic_international_epr": metrics["is_domestic_epr"],
    }

## Baseline prediction

The baseline scores `xendcg_s42_score` are read from `outputs/supernova_validation_predictions.parquet`. If that file is missing, the notebook trains the same model at query weight `1.0` instead.

In [5]:
eval_base = val_df[["srch_id", "prop_id", "booking_bool", "click_bool", "is_domestic"]].copy()
baseline_source = None
baseline_model = None

if BASELINE_PREDICTIONS_PATH.exists():
    saved_predictions = pd.read_parquet(BASELINE_PREDICTIONS_PATH)
    if BASELINE_SCORE_COL in saved_predictions.columns:
        baseline_pred = saved_predictions[["srch_id", "prop_id", BASELINE_SCORE_COL]].copy()
        baseline_source = str(BASELINE_PREDICTIONS_PATH)
    else:
        print(f"Saved predictions at {BASELINE_PREDICTIONS_PATH} lack {BASELINE_SCORE_COL!r}")
        baseline_pred = None
else:
    baseline_pred = None

if baseline_pred is None:
    assert TRAIN_BASELINE_IF_SAVED_PREDICTIONS_MISSING, "Need saved baseline predictions or baseline training enabled."
    baseline_model, baseline_pred = train_rankxendcg_with_international_query_weight(
        1.0,
        BASELINE_SCORE_COL,
    )
    baseline_source = "trained_in_diagnostic_notebook"

eval_scores = eval_base.merge(
    baseline_pred,
    on=["srch_id", "prop_id"],
    how="left",
    validate="one_to_one",
)
assert len(eval_scores) == len(eval_base)
assert eval_scores[BASELINE_SCORE_COL].notna().all(), "Baseline predictions did not align with validation rows."

baseline_metrics = domestic_fairness_metrics(eval_scores, BASELINE_SCORE_COL)
summary_rows = [metrics_row("unweighted_seed42_rankxendcg", 1.0, baseline_metrics)]
print(f"Baseline source: {baseline_source}")
pd.DataFrame(summary_rows)

Baseline source: /home/akshitanchan/Projects/hotel-trivago-but-its-expedia/outputs/supernova_validation_predictions.parquet


,name,international_query_weight,overall_ndcg5,overall_recall5,international_ndcg5,domestic_ndcg5,international_recall5,domestic_recall5,domestic_international_epr
0,unweighted_seed42_rankxendcg,1.0,0.418153,0.576976,0.405957,0.425311,0.559727,0.587333,0.954495


## Train weighted models

This cell trains one model for each query weight in the configuration and records its metrics next to the baseline.

In [6]:
weighted_models = {}
weighted_metrics = {}

for query_weight in INTERNATIONAL_QUERY_WEIGHTS:
    weight_key = str(query_weight).replace(".", "p")
    score_col = f"xendcg_int_query_weight_{weight_key}"
    model, weighted_pred = train_rankxendcg_with_international_query_weight(query_weight, score_col)
    eval_scores = eval_scores.merge(
        weighted_pred,
        on=["srch_id", "prop_id"],
        how="left",
        validate="one_to_one",
    )
    assert eval_scores[score_col].notna().all(), f"Weighted predictions missing for {score_col}"
    run_metrics = domestic_fairness_metrics(eval_scores, score_col)
    weighted_models[score_col] = model
    weighted_metrics[score_col] = run_metrics
    summary_rows.append(metrics_row(score_col, query_weight, run_metrics))

summary_df = pd.DataFrame(summary_rows)
summary_df

xendcg_int_query_weight_1p7: dropout kept 2,639,761/3,966,682 training rows
xendcg_int_query_weight_1p7: international query weight=1.700; weighted training-row mean=1.2529
Training until validation scores don't improve for 250 rounds
[200]	valid_0's ndcg@5: 0.402063
[400]	valid_0's ndcg@5: 0.408673
[600]	valid_0's ndcg@5: 0.412516
[800]	valid_0's ndcg@5: 0.415031
[1000]	valid_0's ndcg@5: 0.415756
[1200]	valid_0's ndcg@5: 0.416226
[1400]	valid_0's ndcg@5: 0.417011
[1600]	valid_0's ndcg@5: 0.416614
Early stopping, best iteration is:
[1378]	valid_0's ndcg@5: 0.417184
xendcg_int_query_weight_1p7: trees=1378


,name,international_query_weight,overall_ndcg5,overall_recall5,international_ndcg5,domestic_ndcg5,international_recall5,domestic_recall5,domestic_international_epr
0,unweighted_seed42_rankxendcg,1.0,0.418153,0.576976,0.405957,0.425311,0.559727,0.587333,0.954495
1,xendcg_int_query_weight_1p7,1.7,0.416735,0.576188,0.404427,0.423958,0.559367,0.586289,0.953930


## Results

The next cell writes `outputs/supernova_bias_weighted_international_run.json` and prints it. The JSON holds the summary table, detailed domestic and international metrics, and the number of validation searches whose candidate set mixes domestic and international hotels.

In [7]:
domestic_nunique = eval_base.groupby("srch_id")["is_domestic"].nunique()
mixed_domestic_queries = int((domestic_nunique > 1).sum())

payload = {
    "diagnostic_tag": DIAGNOSTIC_TAG,
    "baseline_source": baseline_source,
    "feature_count": int(len(feature_cols)),
    "row_count": int(len(eval_base)),
    "search_count": int(eval_base["srch_id"].nunique()),
    "mixed_domestic_queries": {
        "count": mixed_domestic_queries,
        "fraction": float(mixed_domestic_queries / eval_base["srch_id"].nunique()),
    },
    "summary": summary_df.to_dict(orient="records"),
    "baseline_metrics": baseline_metrics,
    "weighted_metrics": weighted_metrics,
}

output_path = OUTPUT_DIR / f"{DIAGNOSTIC_TAG}_run.json"
save_run_json(output_path, payload)
print(f"Wrote {output_path}")
print(json.dumps(payload, indent=2))

Wrote /home/akshitanchan/Projects/hotel-trivago-but-its-expedia/outputs/supernova_bias_weighted_international_run.json
{
  "diagnostic_tag": "supernova_bias_weighted_international",
  "baseline_source": "/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/outputs/supernova_validation_predictions.parquet",
  "feature_count": 273,
  "row_count": 991665,
  "search_count": 39959,
  "mixed_domestic_queries": {
    "count": 68,
    "fraction": 0.0017017442878950926
  },
  "summary": [
    {
      "name": "unweighted_seed42_rankxendcg",
      "international_query_weight": 1.0,
      "overall_ndcg5": 0.41815315145764154,
      "overall_recall5": 0.5769758599743526,
      "international_ndcg5": 0.40595739398981184,
      "domestic_ndcg5": 0.4253112570818296,
      "international_recall5": 0.5597265531302471,
      "domestic_recall5": 0.5873330212796601,
      "domestic_international_epr": 0.954494825213869
    },
    {
      "name": "xendcg_int_query_weight_1p7",
      "international_quer